# 13 Transformer

**Parameters:**
* `src_vocab_size`: $\;$ size of the input vocabulary
* `tgt_vocab_size`: $\;$ size of the output vocabulary
* `embed_dim`: $\;$ embedding dimension (both input and output), $\,d_{\text{model}}=512\,$ in article
* `num_layers`: $\;$ number of encoder and decoder layers, $\,N=6\,$ in article
* `num_heads`: $\;$ number of heads in multihead handling, $\,h=8\,$ in article
* `dropout=0.1`: $\;$ dropout probability, $\,P_{\text{drop}}=0.1\,$ in article
* `pre_norm=True`: $\;$ LayerNorm first; after Add in article
* `pe_type='fixed'`: $\;$ fixed or trainable positional encoding; fixed in article, although they try to train

**`Generator:`** $\;$ auxiliary class that implements the linear head with an unbiased linear transformation

In [1]:
%%script true
class Generator(nn.Module):
    """
    Define decoder side language model head (`final_proj`, a linear projection) and softmax for generation.
    Note: To tie the weights of final_proj with nn.Embedding, bias term in final_proj is set to be False.
    """
    def __init__(self, embed_dim, vocab_size):
        super().__init__(); self.final_proj = nn.Linear(embed_dim, vocab_size, bias=False) # projection to voc size    
    def forward(self, x):
        return F.log_softmax(self.final_proj(x), dim=-1) # softmax and log


<p style="page-break-after:always;"></p>


**`Transformer`:** $\;$ subclass of `nn.Module`
* **Instantiation:** $\;$ initializes embeddings, encoder, decoder, and `weight` head tied to the output embedding's weight

In [2]:
%%script true
self.src_embed = EmbeddingsWithPositionalEncoding(src_vocab_size, embed_dim, dropout, pe_type)
self.tgt_embed = EmbeddingsWithPositionalEncoding(tgt_vocab_size, embed_dim, dropout, pe_type)
self.encoder = Encoder(embed_dim, num_layers, num_heads, dropout, pre_norm)
self.decoder = Decoder(embed_dim, num_layers, num_heads, dropout, pre_norm)
self.generator = Generator(embed_dim, tgt_vocab_size)
self.generator.final_proj.weight = self.tgt_embed.embed.weight

* **`encode`:** $\;$ implements $\,\operatorname{Encoder}(X)\,$ after getting the embedding of the input, $\,X$

In [3]:
%%script true
def encode(self, src, src_mask):
    x = self.src_embed(src) # embedding of src
    enc_out = self.encoder(x, src_mask) # encoder output
    return enc_out

* **`decode`:** $\;$ implements $\,\operatorname{Decoder}(X, Y, M)\,$ after getting the output embedding, $\,Y$

In [4]:
%%script true
def decode(self, tgt, memory, src_mask, tgt_mask):  # the order of args should be consistent across modules
    x = self.tgt_embed(tgt) # embedding of tgt
    dec_out = self.decoder(x, memory, src_mask, tgt_mask) # 'encoder output' serves as K,V in 'decoder x-attention'
    return dec_out

* **`forward`:** $\;$ implements $\operatorname{Transformer}(X, Y, M)=\operatorname{Decoder}(\operatorname{Encoder}(X), Y, M)$

In [5]:
%%script true
def forward(self, src, tgt, src_mask, tgt_mask):
    memory = self.encode(src, src_mask) # encoder output (memory)
    dec_out = self.decode(tgt, memory, src_mask, tgt_mask) # decoder output
    return dec_out


* **`tie_weight`:** $\;$ ties the weight of the input embedding with the output embedding (e.g., if they share vocs)

In [6]:
%%script true
def tie_weights(self):
    self.src_embed.embed.weight = self.tgt_embed.embed.weight
    print("Source (encoder) and target (decoder) embedding weights are now tied.")

* **Attribute `device`:** $\;$ determines the device on which the model is located

In [7]:
%%script true
@property
def device(self) -> torch.device:
    return next(self.parameters()).device

**Creation and random initialization of a model:** $\;$ function used in the examples seen above

In [8]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import Transformer, create_causal_mask

In [9]:
def create_model(src_vocab_size, tgt_vocab_size, embed_dim=512, num_layers=6, num_heads=8, dropout=0.1,
    pre_norm=True, pe_type='fixed', device=None):
    model = Transformer(src_vocab_size, tgt_vocab_size, embed_dim, num_layers, num_heads, dropout,
        pre_norm, pe_type)
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)    
    if device is not None:      # note: `0` indicates `cuda:0`
        model = model.to(device)
    return model


<p style="page-break-after:always;"></p>


**Quick inference test to test the code:**

In [10]:
def inference_test():
    test_model = create_model(src_vocab_size=11, tgt_vocab_size=11, embed_dim=512, num_layers=2, num_heads=8,
        dropout=0.1, pre_norm=True, pe_type='fixed', device=None)
    # do not tie the weight for a random test:
    test_model.generator.final_proj.weight = nn.Parameter(torch.randn_like(test_model.generator.final_proj.weight))
    test_model.eval()
    src = torch.randint(1, 11, (1, 10)); src_mask = torch.ones(1, 1, 10)
    memory = test_model.encode(src, src_mask); ys = torch.zeros(1, 1).type_as(src) # ys[0]=0    
    # model rollout
    for i in range(9):
        tgt_mask = create_causal_mask(ys.size(1)).type_as(src.data)
        out = test_model.decode(ys, memory, src_mask, tgt_mask)
        prob = test_model.generator(out[:, -1])     # last token
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    print(f"{src} -> {ys}")

In [11]:
for _ in range(10):
    inference_test()

tensor([[ 7,  7,  1,  3,  7, 10,  5,  8,  1,  4]]) -> tensor([[0, 6, 3, 6, 3, 6, 3, 6, 3, 9]])
tensor([[5, 6, 1, 9, 6, 4, 8, 8, 9, 3]]) -> tensor([[0, 1, 7, 7, 7, 7, 7, 7, 7, 7]])
tensor([[ 5,  9,  6,  7,  8,  6,  1,  2,  9, 10]]) -> tensor([[0, 9, 9, 9, 9, 9, 0, 0, 0, 0]])
tensor([[ 9,  7,  2,  1, 10, 10, 10,  9,  3,  2]]) -> tensor([[0, 4, 1, 1, 1, 1, 1, 1, 1, 1]])
tensor([[3, 1, 8, 9, 4, 3, 9, 4, 3, 8]]) -> tensor([[0, 1, 6, 1, 8, 3, 3, 3, 3, 3]])
tensor([[ 9,  6,  6,  2,  9,  1, 10,  9, 10,  5]]) -> tensor([[ 0,  9,  5,  0,  9, 10,  0,  9, 10,  0]])
tensor([[10, 10,  6,  5,  3,  5,  6,  8,  8,  4]]) -> tensor([[0, 9, 8, 4, 9, 8, 8, 8, 9, 8]])
tensor([[4, 2, 5, 4, 9, 6, 8, 2, 3, 5]]) -> tensor([[0, 4, 6, 3, 4, 6, 3, 0, 7, 7]])
tensor([[7, 2, 1, 4, 3, 6, 4, 8, 5, 2]]) -> tensor([[0, 8, 7, 7, 7, 7, 7, 7, 7, 7]])
tensor([[5, 2, 9, 2, 7, 7, 4, 7, 8, 3]]) -> tensor([[ 0,  7,  1,  2,  0, 10, 10, 10, 10, 10]])



<p style="page-break-after:always;"></p>
